<a href="https://colab.research.google.com/github/domaamohamed00-ui/8_queens-algorithms-/blob/main/imdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install datasets -q

In [ ]:
from datasets import load_dataset
dataset = load_dataset("stanfordnlp/imdb")
print(dataset)
print(dataset["train"][0])

In [ ]:
import re #regular expression
from collections import Counter
def tokenize(text):
  text = text.lower()
  text = re.sub(r"[^a-zA-Z0-9]", " ", text)
  return text.split()

sample_text = dataset["train"][0]["text"]
tokens = tokenize(sample_text)
print(tokens[:20])

In [ ]:
from collections import Counter

word_counter = Counter()
count = 0

for example in dataset["train"].select(range(5000)):
    tokens = tokenize(example["text"])
    word_counter.update(tokens)
    count += 1

print(word_counter.most_common(10))
len(word_counter)

In [ ]:
MAX_VOCAB_SIZE = 10000
most_common_words = word_counter.most_common(MAX_VOCAB_SIZE)
vocab = {"<pad>": 0, "<unk>": 1}
for word,freq in most_common_words:
    vocab[word] = len(vocab)
print(len(vocab))
print(vocab["<pad>"])
print(vocab["<unk>"])

In [ ]:
MAX_LEN = 200  # هنحدد أقصى طول للجملة (200 كلمة)، أي جملة أطول هنقصها، وأي جملة أقصر هنعمل لها padding

def encode_text(text, vocab, max_len=MAX_LEN):
    tokens = tokenize(text)
    # نحول كل كلمة لرقمها من الـ vocab، ولو مش موجودة نحطها <unk>
    ids = [vocab.get(token, vocab["<unk>"]) for token in tokens]

    if len(ids) > max_len:
        ids = ids[:max_len]  # نقص الزيادة
    else:
        ids = ids + [vocab["<pad>"]] * (max_len - len(ids))  # نكمل بـ padding لحد ما تبقى 200

    return ids

# نجرب الدالة على أول مراجعة
sample_encoded = encode_text(dataset["train"][0]["text"], vocab)
print(sample_encoded[:30])
print( len(sample_encoded))

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# هنستخدم أول 5000 من train للتدريب و 2000 من test للتقييم (عشان السرعة دلوقتي)
TRAIN_SIZE = 15000
TEST_SIZE = 5000

train_data = dataset["train"].select(range(TRAIN_SIZE))
test_data = dataset["test"].select(range(TEST_SIZE))

# نعمل كلاس Dataset مخصوص عشان نقدر نستخدمه مع DataLoader
class IMDBDataset(Dataset):
    def __init__(self, hf_dataset, vocab, max_len=MAX_LEN):
        self.hf_dataset = hf_dataset
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        example = self.hf_dataset[idx]
        ids = encode_text(example["text"], self.vocab, self.max_len)
        label = example["label"]
        return torch.tensor(ids, dtype=torch.long), torch.tensor(label, dtype=torch.float)

# نبني الـ Dataset objects
train_dataset = IMDBDataset(train_data, vocab)
test_dataset = IMDBDataset(test_data, vocab)

# نبني الـ DataLoader (هيقسملنا الداتا لـ batches تلقائي)
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# نجرب ناخد batch واحد ونشوف شكله
sample_batch = next(iter(train_loader))
texts, labels = sample_batch
print("texts:", texts.shape)
print("labels:", labels.shape)

In [ ]:
import torch.nn as nn

class LSTMSentimentModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        last_hidden = hidden[-1]
        out = self.fc(last_hidden)
        out = self.sigmoid(out)
        return out.squeeze(1)

In [ ]:
import torch.optim as optim

# 1. تحديد حجم الـ Hyperparameters والموديل
VOCAB_SIZE = len(vocab)
EMBEDDING_DIM = 64
HIDDEN_DIM = 128
LEARNING_RATE = 0.001
EPOCHS = 3

# إنشاء كائن الموديل
model = LSTMSentimentModel(vocab_size=VOCAB_SIZE, embedding_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_DIM)

# دالة الخسارة والمُحسّن
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for texts, labels in train_loader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        predictions = model(texts)
        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {avg_loss:.4f}")

In [ ]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for texts, labels in test_loader:
        texts, labels = texts.to(device), labels.to(device)
        predictions = model(texts)
        predicted_labels = (predictions >= 0.5).float()
        correct += (predicted_labels == labels).sum().item()
        total += labels.size(0)
accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:
# جرب الموديل على جملة من عندك مش من الداتاسيت خالص
test_sentence = "the boy run slow"
encoded = encode_text(test_sentence, vocab)
input_tensor = torch.tensor(encoded).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    prediction = model(input_tensor)
    print("الاحتمال:", prediction.item())
    print("التصنيف:", "إيجابي" if prediction.item() >= 0.5 else "سلبي")

In [ ]:
# نتأكد الكلمات موجودة في الـ vocab ولا لأ
words = ["this", "movie", "was", "happy"]
for w in words:
    print(w, "->", vocab.get(w, "مش موجودة (unk)"))